In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

In [2]:
# --- Data Preparation ---
if not Path ("input.txt").exists():
    import urllib.request
    print("downloading")
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)} #String to Index
itos = {i: ch for ch, i in stoi.items()} #Index to String
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

downloading


In [3]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size #How many characters the model look at once.

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # x and y are both sequences of length block_size
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

In [4]:
# --- Transformer Component ---
class Head(nn.Module):
    """ One head of masked self-attention """
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        # Compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5) # (B, T, T)
        # Causal mask: prevent tokens from looking into the future
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Perform the weighted aggregation of values
        out = wei @ v # (B, T, head_size)
        return out

In [5]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention running in parallel """
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Concatenate outputs from all heads along the channel/embedding dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out) # Projection back into the residual path
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity """
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communicates (attention) then computes (feedforward) """
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        # Pre-LN combined with Residual connections
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [7]:
# -- TinyGPT ---
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=256, num_heads=8, num_layers=8, dropout=0.1):  #Increased Capacity to increase long-term sentences
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)

        tok_emb = self.token_embedding(x)      # (B, T, emb_dim)
        pos_emb = self.position_embedding(pos) # (T, emb_dim)

        h = tok_emb + pos_emb                  # (B, T, emb_dim)
        h = self.blocks(h)                     # (B, T, emb_dim)
        h = self.ln_f(h)                       # (B, T, emb_dim)
        logits = self.lm_head(h)               # (B, T, vocab_size)
        return logits

# Visual For How It Calculates
These parameters are not optimal for the model, this is only done to visually see how it calculates and predicts the next letter.

In [28]:
import torch
import torch.nn.functional as F

# Define block_size and device here for debugging purposes
block_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Set up hyperparameters (match model instantiation in main block)
emb_dim = 8
num_heads = 4
num_layers = 4
dropout = 0.1

# Instantiate the model
model = TinyGPT(vocab_size, block_size, emb_dim, num_heads, num_layers, dropout).to(device)

model.eval() # Set to evaluation mode to disable dropout, etc.

# Get the first character's index
first_char_idx = data[0].item()
x_input = torch.tensor([[first_char_idx]], device=device) # (B=1, T=1)

# Get the first word from the original text for clarity
first_word_from_text = text.split()[0]
print(f"The actual first word from the raw text is: '{first_word_from_text}'\n")

print(f"Index of the first character the model processes: {first_char_idx}")
print(f"The first character the model processes is: '{itos[first_char_idx]}'\n")

print(f"--- Processing the first token: '{itos[first_char_idx]}' (index {first_char_idx}) ---")
print(f"Input tensor shape: {x_input.shape}\n")

with torch.no_grad():
    # 1. TinyGPT: token_embedding
    tok_emb = model.token_embedding(x_input)
    print(f"TinyGPT: Token Embedding Output Shape: {tok_emb.shape}\nTensor:\n{tok_emb}\n")

    # 1. TinyGPT: position_embedding
    # For a single token, pos will be [0]
    pos = torch.arange(x_input.shape[1], device=x_input.device)
    pos_emb = model.position_embedding(pos)
    print(f"TinyGPT: Position Embedding Output Shape: {pos_emb.shape}\nTensor:\n{pos_emb}\n")

    # 1. TinyGPT: initial h (input to blocks)
    h = tok_emb + pos_emb
    print(f"TinyGPT: Initial 'h' (Token + Pos Emb) Shape: {h.shape}\nTensor:\n{h}\n")

Using device: cuda
The actual first word from the raw text is: 'First'

Index of the first character the model processes: 18
The first character the model processes is: 'F'

--- Processing the first token: 'F' (index 18) ---
Input tensor shape: torch.Size([1, 1])

TinyGPT: Token Embedding Output Shape: torch.Size([1, 1, 8])
Tensor:
tensor([[[ 0.8899,  0.2221, -0.2738,  0.1161, -1.2907,  1.5031,  0.9740,
           0.9111]]], device='cuda:0')

TinyGPT: Position Embedding Output Shape: torch.Size([1, 8])
Tensor:
tensor([[ 0.7639,  0.4541,  0.1459,  0.5193, -0.8206, -2.1833, -0.9255, -0.3889]],
       device='cuda:0')

TinyGPT: Initial 'h' (Token + Pos Emb) Shape: torch.Size([1, 1, 8])
Tensor:
tensor([[[ 1.6538,  0.6762, -0.1279,  0.6354, -2.1113, -0.6802,  0.0485,
           0.5222]]], device='cuda:0')



In [29]:
import torch
import torch.nn.functional as F

# Reuse the 'h' tensor from the previous cell, which holds the initial token + position embeddings
# It's also important to ensure 'model', 'itos', 'first_char_idx' are still in scope or passed as args

# 2. Iterate through Blocks
with torch.no_grad():
    for i, block in enumerate(model.blocks):
        print(f"\n--- Block {i+1}/{len(model.blocks)} ---")
        x_block_input = h # Input to the current block

        # LayerNorm 1
        ln1_out = block.ln1(x_block_input)
        print(f"Block {i+1}: LayerNorm1 Output Shape: {ln1_out.shape}\nTensor:\n{ln1_out}\n")

        # MultiHeadAttention
        mha = block.sa
        print(f"Block {i+1}: MultiHeadAttention (SA) Input Shape: {ln1_out.shape}")

        head_outputs = []
        for j, head in enumerate(mha.heads):
            print(f"  MultiHeadAttention: Head {j+1}/{len(mha.heads)}")

            k = head.key(ln1_out)
            q = head.query(ln1_out)
            v = head.value(ln1_out)
            print(f"    Head {j+1}: Key (k) Shape: {k.shape}\n    Tensor:\n{k}\n")
            print(f"    Head {j+1}: Query (q) Shape: {q.shape}\n    Tensor:\n{q}\n")
            print(f"    Head {j+1}: Value (v) Shape: {v.shape}\n    Tensor:\n{v}\n")

            wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
            print(f"    Head {j+1}: Attention Scores (wei before softmax) Shape: {wei.shape}\n    Tensor:\n{wei}\n")

            # For a single token (T=1), tril is [[1]], so masked_fill will not change anything
            # wei = wei.masked_fill(head.tril[:x_input.shape[1], :x_input.shape[1]] == 0, float("-inf"))

            wei = F.softmax(wei, dim=-1)
            print(f"    Head {j+1}: Attention Weights (wei after softmax) Shape: {wei.shape}\n    Tensor:\n{wei}\n")

            head_out = wei @ v
            print(f"    Head {j+1}: Head Output Shape: {head_out.shape}\n    Tensor:\n{head_out}\n")
            head_outputs.append(head_out)

        mha_concat_out = torch.cat(head_outputs, dim=-1)
        print(f"MultiHeadAttention: Concatenated Heads Output Shape: {mha_concat_out.shape}\nTensor:\n{mha_concat_out}\n")

        sa_out = mha.proj(mha_concat_out)
        print(f"MultiHeadAttention: Projection (proj) Output Shape: {sa_out.shape}\nTensor:\n{sa_out}\n")

        # Residual connection 1
        h = x_block_input + sa_out
        print(f"Block {i+1}: Residual Connection 1 Output (after SA) Shape: {h.shape}\nTensor:\n{h}\n")

        # LayerNorm 2
        ln2_out = block.ln2(h)
        print(f"Block {i+1}: LayerNorm2 Output Shape: {ln2_out.shape}\nTensor:\n{ln2_out}\n")

        # FeedForward
        ffwd = block.ffwd
        print(f"Block {i+1}: FeedForward Input Shape: {ln2_out.shape}")

        ffwd_linear1_out = ffwd.net[0](ln2_out)
        print(f"  FeedForward: First Linear Layer Output Shape: {ffwd_linear1_out.shape}\n  Tensor:\n{ffwd_linear1_out}\n")

        ffwd_relu_out = ffwd.net[1](ffwd_linear1_out)
        print(f"  FeedForward: ReLU Output Shape: {ffwd_relu_out.shape}\n  Tensor:\n{ffwd_relu_out}\n")

        ffwd_linear2_out = ffwd.net[2](ffwd_relu_out)
        print(f"  FeedForward: Second Linear Layer Output Shape: {ffwd_linear2_out.shape}\n  Tensor:\n{ffwd_linear2_out}\n")

        ffwd_out = ffwd_linear2_out
        print(f"FeedForward: Final Output Shape: {ffwd_out.shape}\nTensor:\n{ffwd_out}\n")

        # Residual connection 2
        h = h + ffwd_out
        print(f"Block {i+1}: Final Block Output (after Residual Connection 2) Shape: {h.shape}\nTensor:\n{h}\n")


--- Block 1/4 ---
Block 1: LayerNorm1 Output Shape: torch.Size([1, 1, 8])
Tensor:
tensor([[[ 1.5067,  0.5726, -0.1959,  0.5335, -2.0913, -0.7237, -0.0273,
           0.4254]]], device='cuda:0')

Block 1: MultiHeadAttention (SA) Input Shape: torch.Size([1, 1, 8])
  MultiHeadAttention: Head 1/4
    Head 1: Key (k) Shape: torch.Size([1, 1, 2])
    Tensor:
tensor([[[0.6122, 0.2445]]], device='cuda:0')

    Head 1: Query (q) Shape: torch.Size([1, 1, 2])
    Tensor:
tensor([[[-0.4298,  0.9588]]], device='cuda:0')

    Head 1: Value (v) Shape: torch.Size([1, 1, 2])
    Tensor:
tensor([[[-0.6574,  1.0832]]], device='cuda:0')

    Head 1: Attention Scores (wei before softmax) Shape: torch.Size([1, 1, 1])
    Tensor:
tensor([[[-0.0203]]], device='cuda:0')

    Head 1: Attention Weights (wei after softmax) Shape: torch.Size([1, 1, 1])
    Tensor:
tensor([[[1.]]], device='cuda:0')

    Head 1: Head Output Shape: torch.Size([1, 1, 2])
    Tensor:
tensor([[[-0.6574,  1.0832]]], device='cuda:0')

  

In [30]:
    # 3. TinyGPT: final layers
    ln_f_out = model.ln_f(h)
    print(f"TinyGPT: Final LayerNorm (ln_f) Output Shape: {ln_f_out.shape}\nTensor:\n{ln_f_out}\n")

    logits = model.lm_head(ln_f_out)
    print(f"TinyGPT: Language Model Head (lm_head) Logits Output Shape: {logits.shape}\nTensor:\n{logits}\n")

    # Optional: Softmax to get probabilities for the next token
    probs = F.softmax(logits, dim=-1)
    print(f"TinyGPT: Probabilities (softmax over logits) Shape: {probs.shape}\n")
    print(f"Top 5 predicted next tokens for '{itos[first_char_idx]}':")
    top_probs, top_indices = torch.topk(probs, 5, dim=-1)
    for k in range(5):
        print(f"  {k+1}. '{itos[top_indices[0, 0, k].item()]}' (index {top_indices[0, 0, k].item()}) with probability {top_probs[0, 0, k].item():.4f}")

TinyGPT: Final LayerNorm (ln_f) Output Shape: torch.Size([1, 1, 8])
Tensor:
tensor([[[ 0.9713,  0.4419, -0.2487, -0.7419, -1.9359, -0.5703,  0.9908,
           1.0928]]], device='cuda:0', grad_fn=<NativeLayerNormBackward0>)

TinyGPT: Language Model Head (lm_head) Logits Output Shape: torch.Size([1, 1, 65])
Tensor:
tensor([[[-0.3698,  0.3457, -0.0940,  1.3989, -0.0910,  0.2333, -0.2470,
           0.2503,  0.1710,  0.0775,  0.5993, -0.0364,  0.3137, -0.3317,
           0.3844, -0.3336, -0.1000, -0.5404,  0.2902, -0.9496,  0.0794,
          -0.9740,  0.2124,  0.3263, -1.6942, -0.1041,  0.0652, -0.1342,
           0.1066, -1.0218, -0.1846,  0.4395, -0.4153,  0.5003, -0.5649,
           0.8249,  0.2339,  0.5313,  0.2845, -0.2451,  0.1604,  0.2274,
          -0.3524,  0.4870, -0.8743,  0.0283,  0.2941,  0.3770,  0.7331,
          -0.9417,  0.5070,  0.1187,  0.2280, -0.0317,  0.4326, -0.7200,
          -0.4203,  0.8206, -0.1996,  0.5549, -0.2736, -0.3806,  0.6588,
           0.6038, -0.5832]

# Training the Model

In [32]:
# --- Training Loop + Evaluation ---

def sequence_cross_entropy(logits, targets):
    # Reshape targets to (B*T) and logits to (B*T, V) for standard cross entropy
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)

        if max_steps is not None and step + 1 >= max_steps:
            break

    return total_loss / total_count

In [33]:
# --- Sampling and Text Generation ---

@torch.no_grad()
def sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=200):
    model.eval()
    # Initialize a blank context window filled with 0s
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)

    # Pre-fill context with your prompt characters
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)

    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :] # Crop logits to focus only on the last step prediction
        probs = F.softmax(logits, dim=-1)

        # Sample using multinomial distribution instead of argmax for varied generation
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])

        # Shift the context window over by 1 to make room for the new token
        context = torch.cat([context[:, 1:], ix], dim=1)

    return "".join(out)

In [34]:
# --- Execution ---

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Hyperparameters
    block_size = 64
    batch_size = 64
    epochs = 100         # Adjust this up to 50 or 100 once you verify it works!
    max_steps_per_epoch = 150

    # Dataset setups
    dataset = NextTokenDataset(data, block_size)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Model instantiation
    model = TinyGPT(vocab_size, block_size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    print("Starting training script...")
    for epoch in range(epochs):
        loss = train_one_epoch(model, loader, optimizer, device, max_steps=max_steps_per_epoch)
        print(f"Epoch {epoch:2d} | Train Loss {loss:.4f}")

    print("\n--- GENERATED TEXT TEST ---")
    generated_sample = sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:\n", max_new_tokens=300)
    print(generated_sample)

Using device: cuda
Starting training script...
Epoch  0 | Train Loss 2.5979
Epoch  1 | Train Loss 2.2434
Epoch  2 | Train Loss 2.0367
Epoch  3 | Train Loss 1.8864
Epoch  4 | Train Loss 1.7861
Epoch  5 | Train Loss 1.7126
Epoch  6 | Train Loss 1.6547
Epoch  7 | Train Loss 1.6156
Epoch  8 | Train Loss 1.5812
Epoch  9 | Train Loss 1.5507
Epoch 10 | Train Loss 1.5219
Epoch 11 | Train Loss 1.5035
Epoch 12 | Train Loss 1.4911
Epoch 13 | Train Loss 1.4689
Epoch 14 | Train Loss 1.4548
Epoch 15 | Train Loss 1.4400
Epoch 16 | Train Loss 1.4335
Epoch 17 | Train Loss 1.4208
Epoch 18 | Train Loss 1.4122
Epoch 19 | Train Loss 1.4024
Epoch 20 | Train Loss 1.3887
Epoch 21 | Train Loss 1.3806
Epoch 22 | Train Loss 1.3758
Epoch 23 | Train Loss 1.3688
Epoch 24 | Train Loss 1.3565
Epoch 25 | Train Loss 1.3546
Epoch 26 | Train Loss 1.3468
Epoch 27 | Train Loss 1.3377
Epoch 28 | Train Loss 1.3362
Epoch 29 | Train Loss 1.3314
Epoch 30 | Train Loss 1.3169
Epoch 31 | Train Loss 1.3203
Epoch 32 | Train Loss 1.3